### EV Resale Price Regression

In [ ]:
import pandas as pd
df = pd.read_csv("C:\\Users\\DELL\\Downloads\\EV_Resale_Price_Regression - EV_Resale_Price_Regression.csv")


In [5]:
# 1. Data Quality Audit
#Perform a complete audit of the dataset:
#• shape
#• datatypes
#• missing values
#• duplicate records
#• unique vehicle IDs
#• categorical distributions
#Identify every data-quality problem before modifying the dataset.

df.shape[0]
df.shape[1]
df.shape

(1000, 15)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   vehicle_id            1000 non-null   object 
 1   listing_date          1000 non-null   object 
 2   manufacture_year      1000 non-null   int64  
 3   brand                 1000 non-null   object 
 4   vehicle_type          1000 non-null   object 
 5   battery_capacity_kwh  975 non-null    float64
 6   battery_health_pct    970 non-null    float64
 7   range_km              980 non-null    float64
 8   km_driven             1000 non-null   float64
 9   charging_time_hr      976 non-null    float64
 10  fast_charging         1000 non-null   object 
 11  owner_count           1000 non-null   int64  
 12  city                  1000 non-null   object 
 13  service_history       970 non-null    object 
 14  resale_price          1000 non-null   int64  
dtypes: float64(5), int64(3

In [7]:
df.dtypes

vehicle_id               object
listing_date             object
manufacture_year          int64
brand                    object
vehicle_type             object
battery_capacity_kwh    float64
battery_health_pct      float64
range_km                float64
km_driven               float64
charging_time_hr        float64
fast_charging            object
owner_count               int64
city                     object
service_history          object
resale_price              int64
dtype: object

In [8]:
df.isnull().sum()

vehicle_id               0
listing_date             0
manufacture_year         0
brand                    0
vehicle_type             0
battery_capacity_kwh    25
battery_health_pct      30
range_km                20
km_driven                0
charging_time_hr        24
fast_charging            0
owner_count              0
city                     0
service_history         30
resale_price             0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Length: 1000, dtype: bool

In [11]:
df.columns.tolist()

['vehicle_id',
 'listing_date',
 'manufacture_year',
 'brand',
 'vehicle_type',
 'battery_capacity_kwh',
 'battery_health_pct',
 'range_km',
 'km_driven',
 'charging_time_hr',
 'fast_charging',
 'owner_count',
 'city',
 'service_history',
 'resale_price']

In [13]:
df["vehicle_id"].count()
df["vehicle_id"].nunique()
df["vehicle_id"].duplicated().sum()

np.int64(6)

In [14]:
categorical_cols = df.select_dtypes(include="object").columns
categorical_cols.tolist()

['vehicle_id',
 'listing_date',
 'brand',
 'vehicle_type',
 'fast_charging',
 'city',
 'service_history']

In [20]:
# 2. Duplicate Vehicle Investigation
#vehicle_id is expected to uniquely identify a vehicle.
#Determine:
#• how many duplicated vehicle_id values exist
#• which vehicle IDs are duplicated
#• how many records are affected
#Then decide how you would handle these records without blindly using drop_duplicates().

duplicate_ids = df["vehicle_id"].value_counts()
duplicate_ids = duplicate_ids[duplicate_ids > 1]
len(duplicate_ids)

6

In [22]:
duplicate_ids

vehicle_id
EV-20193    2
EV-20544    2
EV-20515    2
EV-20011    2
EV-20653    2
EV-20279    2
Name: count, dtype: int64

In [25]:
affected_records = df["vehicle_id"].isin(duplicate_ids.index).sum()
affected_records

np.int64(12)

In [27]:
duplicated_records = df[df["vehicle_id"].isin(duplicate_ids.index)].sort_values("vehicle_id")
duplicated_records

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price
119,EV-20011,2025-04-30,2016,Nexora,Sedan,57.9,99.5,NaN,40831.0,8.4,Yes,4,Mumbai,Complete,1591429
852,EV-20011,2027-05-03,2016,Nexora,Sedan,57.9,99.5,NaN,40831.0,8.4,Yes,4,Mumbai,Complete,1591429
655,EV-20193,2026-10-18,2023,Atheron,Crossover,31.3,97.2,285.0,33740.0,8.1,Yes,4,Chennai,Complete,1444991
797,EV-20193,2027-03-09,2023,Atheron,Crossover,31.3,97.2,285.0,33740.0,8.1,Yes,4,Chennai,Complete,1444991
816,EV-20279,2027-03-28,2019,E-Motion,SUV,65.6,87.1,381.0,116102.0,6.8,No,2,Bengaluru,Complete,1395622
857,EV-20279,2027-05-08,2019,E-Motion,SUV,65.6,87.1,381.0,116102.0,6.8,No,2,Bengaluru,Complete,1395622
72,EV-20515,2025-03-14,2016,GreenDrive,Hatchback,61.6,100.0,210.0,37482.0,7.0,Yes,1,Mumbai,Complete,1492498
621,EV-20515,2026-09-14,2016,GreenDrive,Hatchback,61.6,100.0,210.0,37482.0,7.0,Yes,1,Mumbai,Complete,1492498
133,EV-20544,2025-05-14,2024,E-Motion,Sedan,38.0,93.9,511.0,67782.0,3.1,Yes,2,Chennai,Complete,1604620
280,EV-20544,2025-10-08,2024,E-Motion,Sedan,38.0,93.9,511.0,67782.0,3.1,Yes,2,Chennai,Complete,1604620


In [30]:
# 3. Date Conversion & Validation
#Convert listing_date into a proper datetime column.
#Then investigate:
#• earliest listing date
#• latest listing date
#• invalid/missing dates
#• whether the date column is suitable for feature engineering.

df["listing_date"].head()

0    2025-01-01
1    2025-01-02
2    2025-01-03
3    2025-01-04
4    2025-01-05
Name: listing_date, dtype: object

In [31]:
df["listing_date"].dtype

dtype('O')

In [36]:
df["listing_date"] = pd.to_datetime(df["listing_date"],errors="coerce")
df["listing_date"].dtype

dtype('<M8[ns]')

In [37]:
df["listing_date"].min()

Timestamp('2025-01-01 00:00:00')

In [38]:
df["listing_date"].max()

Timestamp('2027-09-27 00:00:00')

In [39]:
df["listing_date"].isna().sum()

np.int64(0)

In [41]:
df["listing_date"].notna().sum()

np.int64(1000)

In [42]:
len(df)

1000

In [46]:
# 4. Vehicle Age Feature
#Create vehicle_age using:
#listing year − manufacture year
#Then identify whether any vehicle has:
#• age < 0
#• age = 0
#• unusually high age

df["vehicle_age"] = df["listing_date"].dt.year - df["manufacture_year"]
df[["listing_date","manufacture_year","vehicle_age"]].head()

,listing_date,manufacture_year,vehicle_age
0,2025-01-01,2021,4
1,2025-01-02,2025,0
2,2025-01-03,2019,6
3,2025-01-04,2022,3
4,2025-01-05,2024,1


In [48]:
negative_age = df[df["vehicle_age"] < 0]
len(negative_age)


0

In [49]:
display(negative_age)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age


In [51]:
zero_age=df[df["vehicle_age"] == 0]
len(zero_age)

40

In [52]:
display(zero_age)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,0
8,EV-20708,2025-01-09,2025,Nexora,Sedan,51.9,92.6,522.0,128031.0000,9.2,Yes,2,Pune,Complete,1608566,0
9,EV-20059,2025-01-10,2025,Atheron,Sedan,65.2,95.1,378.0,55670.0000,5.7,Yes,1,Pune,Complete,1729157,0
12,EV-20381,2025-01-13,2025,Nexora,Sedan,60.9,89.4,553.0,46088.0000,7.3,Yes,2,Pune,Missing,1821084,0
19,EV-20301,2025-01-20,2025,E-Motion,Sedan,53.1,95.4,460.0,54736.0000,5.4,Yes,3,Bengaluru,Complete,1820168,0
22,EV-20842,2025-01-23,2025,Atheron,Crossover,50.1,93.7,193.0,73703.0000,6.1,Yes,1,Chennai,Complete,1438113,0
31,EV-20891,2025-02-01,2025,Voltix,Hatchback,28.7,90.4,370.0,17313.0000,7.9,Yes,1,Pune,Partial,1467882,0
65,EV-20356,2025-03-07,2025,Atheron,Crossover,47.6,94.9,303.0,44227.0000,8.5,No,1,Delhi,Complete,1653416,0
91,EV-20607,2025-04-02,2025,Voltix,Sedan,71.5,91.2,402.0,13806.0000,5.8,Yes,4,Pune,Complete,1856159,0
134,EV-20646,2025-05-15,2025,Voltix,Sedan,49.2,NaN,397.0,118969.0000,6.0,Yes,1,Hyderabad,Partial,1471916,0


In [53]:
df["vehicle_age"].describe()

count    1000.000000
mean        5.426000
std         2.992571
min         0.000000
25%         3.000000
50%         6.000000
75%         8.000000
max        11.000000
Name: vehicle_age, dtype: float64

In [55]:
# 5. Battery Data Imputation
#The following columns contain missing values:
#battery_capacity_kwh, battery_health_pct, range_km, charging_time_hr
#Develop an appropriate missing-value strategy for each column.
#Do not automatically use the same statistic for every column.
#Explain why you selected mean, median, or another strategy.

battery_cols = ["battery_capacity_kwh","battery_health_pct","range_km","charging_time_hr"]
df[battery_cols].isnull().sum()

battery_capacity_kwh    25
battery_health_pct      30
range_km                20
charging_time_hr        24
dtype: int64

In [56]:
df[battery_cols].describe()

,battery_capacity_kwh,battery_health_pct,range_km,charging_time_hr
count,975.000000,970.000000,980.000000,976.000000
mean,58.288755,91.221546,370.018367,6.125000
std,14.199822,4.835603,84.986135,1.755376
min,25.000000,75.800000,150.000000,2.000000
25%,48.700000,87.900000,310.750000,4.900000
50%,58.200000,91.200000,371.000000,6.250000
75%,67.700000,94.600000,427.250000,7.300000
max,110.405980,100.000000,630.000000,12.000000


In [57]:
# 6. Battery Health Outlier Investigation
#Analyze battery_health_pct.
#Identify:
#• values below a reasonable minimum
#• values above 100
#• extreme observations
#Then decide whether to remove, cap, or retain suspicious values.

df["battery_health_pct"].describe()

count    970.000000
mean      91.221546
std        4.835603
min       75.800000
25%       87.900000
50%       91.200000
75%       94.600000
max      100.000000
Name: battery_health_pct, dtype: float64

In [60]:
below_min = df[df["battery_health_pct"] < 0]
len(below_min)

0

In [61]:
display(below_min)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age


In [63]:
above_100 = df[df["battery_health_pct"] > 100]
len(above_100)

0

In [64]:
display(above_100)

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age


In [76]:
q1 = df["battery_health_pct"].quantile(0.25)
q3 = df["battery_health_pct"].quantile(0.75)
IQR = q3 - q1
lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR
q1


np.float64(87.9)

In [75]:
q3

np.float64(8.070697423841708)

In [74]:
IQR

np.float64(2.979527188455579)

In [78]:
lower_bound

np.float64(77.85000000000002)

In [73]:
upper_bound

np.float64(12.539988206525077)

In [71]:
# 7. Range vs Battery Capacity
#Create range_per_kwh using:
#range_km / battery_capacity_kwh
#Identify vehicles with unusually low or high efficiency.

df["range_per_kwh"] = (df["range_km"] / df["battery_capacity_kwh"])
q1 = df["range_per_kwh"].quantile(0.25)
q3 = df["range_per_kwh"].quantile(0.75)
IQR = q3 - q1
lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR
lower_bound

np.float64(0.6218794527027596)

In [72]:
upper_bound

np.float64(12.539988206525077)

In [82]:
# 8. Driving Intensity Feature
#Create km_per_year using:
#km_driven / vehicle_age
#Handle the special case where vehicle_age = 0.
#Identify unusually high annual driving.

(df["vehicle_age"] == 0).sum()

np.int64(40)

In [86]:
df["km_per_year"] = df["km_driven"] / df["vehicle_age"]

In [89]:
q1 = df["km_per_year"].quantile(0.25)
q3 = df["km_per_year"].quantile(0.75)
IQR = q3 - q1
upper_bound = q3 + 1.5 * IQR
upper_bound

np.float64(38548.984375)

In [91]:
# 9. Charging Efficiency
#Create charging_efficiency using:
#range_km / charging_time_hr
#Investigate missing values and extreme values before using this feature in a regression model.

df["charging_efficiency"] = (df["range_km"] / df["charging_time_hr"])
df["charging_time_hr"].isna().sum()

np.int64(24)

In [92]:
df["range_km"].isna().sum()

np.int64(20)

In [93]:
df["charging_efficiency"].isna().sum()

np.int64(44)

In [100]:
mean = df["charging_efficiency"].mean()
std = df["charging_efficiency"].std()
lower_limit = mean - 3*std
upper_limit = mean + 3*std
lower_limit

np.float64(-23.375277655846105)

In [101]:
upper_limit

np.float64(157.3275497507678)

In [102]:
# 10. Ownership Analysis
#Analyze owner_count.
#Determine:
#• frequency of each ownership level
#• whether any values are invalid
#• whether owner count should be treated as numerical or categorical for regression.
#Justify your decision.

df["owner_count"].dtype

dtype('int64')

In [105]:
df["owner_count"].value_counts().sort_index()

owner_count
1    532
2    302
3    126
4     40
Name: count, dtype: int64

In [106]:
(df["owner_count"] == 0).sum

<bound method Series.sum of 0      False
1      False
2      False
3      False
4      False
       ...  
995    False
996    False
997    False
998    False
999    False
Name: owner_count, Length: 1000, dtype: bool>

In [107]:
(df["owner_count"] == 0).sum()

np.int64(0)

In [111]:
df[df["owner_count"] % 1 != 0][["vehicle_id","owner_count"]]

,vehicle_id,owner_count


In [114]:
df["owner_count"] = pd.to_numeric(df["owner_count"],errors ="coerce")
df

,vehicle_id,listing_date,manufacture_year,brand,vehicle_type,battery_capacity_kwh,battery_health_pct,range_km,km_driven,charging_time_hr,fast_charging,owner_count,city,service_history,resale_price,vehicle_age,range_per_kwh,km_per_year,charging_efficiency
0,EV-20170,2025-01-01,2021,Nexora,SUV,46.9,89.6,313.0,28454.0000,10.0,No,1,Chennai,Complete,1527836,4,6.673774,7.113500e+03,31.300000
1,EV-20530,2025-01-02,2025,Nexora,Crossover,80.6,99.5,543.0,183935.8448,7.1,No,2,Chennai,Complete,2040760,0,6.736973,inf,76.478873
2,EV-20654,2025-01-03,2019,Nexora,Crossover,53.9,91.2,NaN,12003.0000,3.6,Yes,1,Chennai,Complete,1830249,6,NaN,2.000500e+03,NaN
3,EV-20935,2025-01-04,2022,GreenDrive,Hatchback,60.1,89.3,528.0,13623.0000,7.8,Yes,2,Delhi,Complete,1726422,3,8.785358,4.541000e+03,67.692308
4,EV-20827,2025-01-05,2024,GreenDrive,Sedan,53.0,92.0,264.0,28548.0000,6.1,No,1,Pune,Complete,1642232,1,4.981132,2.854800e+04,43.278689
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EV-20357,2027-09-23,2017,E-Motion,Sedan,37.9,89.9,333.0,61861.0000,3.0,No,1,Bengaluru,Complete,1338124,10,8.786280,6.186100e+03,111.000000
996,EV-20756,2027-09-24,2020,Atheron,Hatchback,67.5,99.9,290.0,55605.0000,9.2,Yes,1,Bengaluru,Complete,1624596,7,4.296296,7.943571e+03,31.521739
997,EV-20984,2027-09-25,2025,Atheron,Crossover,91.1,81.8,434.0,23708.0000,2.0,No,1,Delhi,Partial,1958519,2,4.763996,1.185400e+04,217.000000
998,EV-20410,2027-09-26,2019,GreenDrive,Sedan,76.8,98.6,264.0,39435.0000,2.8,Yes,1,Chennai,Complete,1724555,8,3.437500,4.929375e+03,94.285714


In [120]:
# 11. Categorical Data Audit
#Analyze: brand, vehicle_type, fast_charging, city, service_history.
#For each column:
#• count unique categories
#• inspect frequencies
#• identify missing values
#• identify suspicious/inconsistent categories.

categorical_cols = ["brand","vehicle_type","fast_charging","city","service_history"]
categorical_cols

['brand', 'vehicle_type', 'fast_charging', 'city', 'service_history']

In [130]:
print(df["brand"].value_counts(dropna=False))

brand
Nexora        213
Atheron       207
GreenDrive    196
E-Motion      192
Voltix        192
Name: count, dtype: int64


In [131]:
print(df["brand"].unique())

['Nexora' 'GreenDrive' 'Atheron' 'E-Motion' 'Voltix']


In [132]:
print(df["brand"].isna().sum())

0


In [135]:
categorical_cols = [
    "brand",
    "vehicle_type",
    "fast_charging",
    "city",
    "service_history"
]
for col in categorical_cols:
    print("\n==============================")
    print("COLUMN:", col)
    print("Unique categories:", df[col].nunique())
    print("Missing values:", df[col].isna().sum())
    print("\nFrequency:")
    print(df[col].value_counts(dropna=False))


COLUMN: brand
Unique categories: 5
Missing values: 0

Frequency:
brand
Nexora        213
Atheron       207
GreenDrive    196
E-Motion      192
Voltix        192
Name: count, dtype: int64

COLUMN: vehicle_type
Unique categories: 4
Missing values: 0

Frequency:
vehicle_type
SUV          265
Hatchback    251
Crossover    244
Sedan        240
Name: count, dtype: int64

COLUMN: fast_charging
Unique categories: 2
Missing values: 0

Frequency:
fast_charging
Yes    692
No     308
Name: count, dtype: int64

COLUMN: city
Unique categories: 6
Missing values: 0

Frequency:
city
Pune         193
Delhi        174
Chennai      168
Bengaluru    160
Mumbai       155
Hyderabad    150
Name: count, dtype: int64

COLUMN: service_history
Unique categories: 3
Missing values: 30

Frequency:
service_history
Complete    590
Partial     297
Missing      83
NaN          30
Name: count, dtype: int64


In [139]:
# 12. Service History Cleaning
#service_history contains Complete, Partial, Missing, and NaN.
#Determine whether “Missing” and actual NaN represent the same business meaning.
#Then create a consistent representation suitable for ML.

(df["service_history"].value_counts(dropna=False))

service_history
Complete    590
Partial     297
Missing      83
NaN          30
Name: count, dtype: int64

In [140]:
(df["service_history"] == "missing").sum()

np.int64(0)

In [141]:
df["service_history"].isna().sum()

np.int64(30)

In [142]:
df = pd.get_dummies(df,columns=["service_history"],dtype=int)

In [143]:
# 13. Fast Charging Transformation
#Transform fast_charging from Yes / No into a binary numerical representation.
#Verify the transformation using frequency counts before and after.

df["fast_charging"].value_counts(dropna=False)

fast_charging
Yes    692
No     308
Name: count, dtype: int64

In [146]:
df["fast_charging"] = df["fast_charging"].map({"Yes":1,"No":0})


In [147]:
df["fast_charging"].value_counts(dropna=False)


fast_charging
NaN    1000
Name: count, dtype: int64

In [148]:
# 14. Target Analysis
#Analyze resale_price.
#Calculate and inspect:
#• mean
#• median
#• minimum
#• maximum
#• standard deviation
#• potential extreme values
#Determine whether the target contains suspicious observations that could strongly influence a regression model.

df["resale_price"].mean()

np.float64(1586535.101)

In [149]:
df["resale_price"].median()

1589594.0

In [150]:
df["resale_price"].min()

915109

In [151]:
df["resale_price"].max()

2072547

In [152]:
df["resale_price"].std()

166204.3759912459

In [155]:
mean = df["resale_price"].mean()
std = df["resale_price"].std()
lower_limit = mean-3*std
upper_limit = mean + 3*std
lower_limit

np.float64(1087921.9730262624)

In [156]:
upper_limit

np.float64(2085148.2289737377)

In [157]:
(df["resale_price"] < 0).sum()

np.int64(0)

In [159]:
(df["resale_price"] == 0).sum()

np.int64(0)

In [161]:
# 15. Price-per-Kilometer Feature
#Create price_per_km using:
#resale_price / km_driven
#Investigate whether extremely low/high values are caused by very low mileage, very high price, or data-quality
#problems.

df["price_per_km"] = df["resale_price"]/df["km_driven"]
print("lowest price per km:")
display(df[["vehicle_id","resale_price","km_driven","price_per_km"]].sort_values("price_per_km").head(10))

lowest price per km:


,vehicle_id,resale_price,km_driven,price_per_km
218,EV-20625,1011244,180000.0000,5.618022
234,EV-20780,915109,149374.0000,6.126294
477,EV-20942,1580186,243880.8896,6.479335
786,EV-20313,1019997,155098.0000,6.576468
51,EV-20007,1207024,180000.0000,6.705689
498,EV-20685,1228000,177346.0000,6.924317
213,EV-20505,1047190,141089.0000,7.422195
587,EV-20244,1306482,175075.0000,7.462413
40,EV-20674,1282088,171158.0000,7.490669
730,EV-20022,1183921,156868.0000,7.547244


In [162]:
print("highest  price per km:")
display(df[["vehicle_id","resale_price","km_driven","price_per_km"]].sort_values("price_per_km",ascending = False).head(10))

highest  price per km:


,vehicle_id,resale_price,km_driven,price_per_km
753,EV-20487,1715613,5779.0,296.870220
362,EV-20854,1866808,6845.0,272.725785
832,EV-20414,1826602,6719.0,271.856229
624,EV-20804,1586813,6086.0,260.731679
702,EV-20946,1981109,8045.0,246.253449
99,EV-20488,1605711,6812.0,235.717998
585,EV-20905,1584806,6965.0,227.538550
7,EV-20584,1589566,6999.0,227.113302
842,EV-20197,1873996,8368.0,223.947897
88,EV-20955,1726590,8232.0,209.741254


In [165]:
low_mileage = df[df["km_driven"] < 1000]
display(low_mileage[["vehicle_id","resale_price","km_driven","price_per_km"]].sort_values("price_per_km",ascending=False))

,vehicle_id,resale_price,km_driven,price_per_km


In [164]:
high_price = df.sort_values("resale_price",ascending=False).head(10)
display(high_price[["vehicle_id","resale_price","km_driven","price_per_km"]])

,vehicle_id,resale_price,km_driven,price_per_km
873,EV-20743,2072547,26173.0000,79.186452
665,EV-20781,2042543,45595.0000,44.797522
1,EV-20530,2040760,183935.8448,11.094955
597,EV-20173,2029110,14720.0000,137.847147
329,EV-20596,2014685,53239.0000,37.842277
216,EV-20024,2012814,36166.0000,55.654869
746,EV-20779,2008096,24123.0000,83.244041
961,EV-20512,1994485,52695.0000,37.849606
445,EV-20682,1993320,18215.0000,109.432885
702,EV-20946,1981109,8045.0000,246.253449


In [166]:
low_price_per_km = df.sort_values("price_per_km").head(10)
display(low_price_per_km[["vehicle_id","resale_price","km_driven","price_per_km"]])

,vehicle_id,resale_price,km_driven,price_per_km
218,EV-20625,1011244,180000.0000,5.618022
234,EV-20780,915109,149374.0000,6.126294
477,EV-20942,1580186,243880.8896,6.479335
786,EV-20313,1019997,155098.0000,6.576468
51,EV-20007,1207024,180000.0000,6.705689
498,EV-20685,1228000,177346.0000,6.924317
213,EV-20505,1047190,141089.0000,7.422195
587,EV-20244,1306482,175075.0000,7.462413
40,EV-20674,1282088,171158.0000,7.490669
730,EV-20022,1183921,156868.0000,7.547244
